In [2]:
import pandas as pd
import numpy as np
import datetime

In [3]:
df= pd.read_csv('C:/Users/Prince_Jaiswal/Capstone1/data/cpg_sales_sample_5000.csv')

In [4]:
df.shape

(5000, 13)

In [5]:
df.head()

,date,store_id,store_region,sku_id,category,units_sold,revenue,promo_flag,promo_type,price,inventory_level,store_size,holiday_flag
0,2022-11-19,3,East,118,Household,9,53.37,0,NaN,5.93,147,Medium,1
1,2022-02-02,1,South,120,Personal Care,7,36.26,0,NaN,5.18,139,Large,0
2,2022-03-26,9,South,149,Snacks,9,71.28,0,NaN,7.92,962,Large,1
3,2022-03-18,9,East,106,Beverages,6,13.32,0,NaN,2.22,371,Medium,0
4,2022-06-08,5,East,146,Household,10,65.50,0,NaN,6.55,229,Large,0


In [6]:
def simulate_price_change(df, price_change_pct=0.15, elasticity=0.5, target_categories=None):
    """
    Simulates a price change and its subsequent impact on demand (units sold) and revenue.
    
    Parameters:
    - df: The baseline pandas DataFrame.
    - price_change_pct: Float, percentage change in price (e.g., 0.15 for 15% increase, -0.1 for 10% decrease).
    - elasticity: Float, demand elasticity coefficient (change in demand = price_change_pct * elasticity).
    - target_categories: List of strings, specific categories to apply the price change to. If None, applies to all.

    Returns:
    A dictionary containing the calculated financial impact metrics:
        - 'baseline_revenue' (float): Total revenue of the original dataset before changes.
        - 'projected_revenue' (float): Total portfolio revenue after combining simulated 
          changes for target categories and baseline values for non-target categories.
        - 'revenue_change_pct' (float): The global percentage change in total revenue 
          compared to the baseline.
        - 'elasticity_coefficient_used' (float): The elasticity factor applied in the simulation.
    """
    df_sim = df.copy()
    baseline_revenue = df_sim['revenue'].sum()
    
    # Create a mask for target rows
    if target_categories:
        mask = df_sim['category'].isin(target_categories)
    else:
        mask = pd.Series(True, index=df_sim.index)
        
    # Apply price hike/change
    df_sim.loc[mask, 'new_price'] = np.round(df_sim.loc[mask, 'price'] * (1 + price_change_pct), 2)
    
    # Calculate demand drop factor based on price change and elasticity
    demand_factor = 1 - (price_change_pct * elasticity)
    
    # Adjust units sold and calculate projected revenue for targeted items
    df_sim.loc[mask, 'projected_units_sold'] = np.maximum(0, np.round(df_sim.loc[mask, 'units_sold'] * demand_factor).astype(int))
    df_sim.loc[mask, 'projected_revenue'] = np.round(df_sim.loc[mask, 'projected_units_sold'] * df_sim.loc[mask, 'new_price'], 2)
    
    # Fill non-targeted rows with their original revenue to ensure a correct global total
    df_sim['projected_revenue'] = df_sim['projected_revenue'].fillna(df_sim['revenue'])
    
    # Calculate total projected revenue across the entire portfolio
    projected_revenue = df_sim['projected_revenue'].sum()
    
    impact = {
        'baseline_revenue': round(baseline_revenue, 2),
        'projected_revenue': round(projected_revenue, 2),
        'revenue_change_pct': round(((projected_revenue - baseline_revenue) / baseline_revenue) * 100, 2),
        'elasticity_coefficient_used': elasticity
    }
    
    return impact

In [7]:
simulate_price_change(df, target_categories=None, price_change_pct=-0.1)

{'baseline_revenue': np.float64(337969.46),
 'projected_revenue': np.float64(317842.46),
 'revenue_change_pct': np.float64(-5.96),
 'elasticity_coefficient_used': 0.5}

In [10]:
def simulate_promo_campaign(df, promo_type='FlashSale', discount_pct=0.20, target_categories=None, conversion_rate=0.30, lift_factor=1.5):
    """
    Simulates introducing a promotion campaign to a subset of currently non-promotional rows.
    
    Parameters:
    - df: The baseline pandas DataFrame.
    - promo_type: String, type of promo to apply ('FlashSale', 'Discount', 'BuyOneGetOne').
    - discount_pct: Float, discount multiplier applied to the baseline price.
    - target_categories: List of strings, the specific categories to run the promo on. If None, applies to all.
    - conversion_rate: Float, fraction of eligible rows to randomly convert to promo status.
    - lift_factor: Float, lift factor to simulate the demand hike in units sold due to promo campaign.

    Returns:
    A dictionary containing the following simulated campaign impact metrics:
        - 'baseline_revenue' (float): Total revenue of the original dataset before changes.
        - 'projected_revenue' (float): Total portfolio revenue after incorporating simulated 
          promo spikes and price reductions for the converted rows.
        - 'revenue_change_pct' (float): The global percentage change in total revenue 
          compared to the baseline.
        - 'discount_pct_used' (float): The discount rate applied to the converted promotional items.
        - 'conversion_rate_used' (float): The target conversion rate used to select promotional items.
        - 'lift_factor_used' (float): The lift factor used to simulate promotional demand spike in units sold.
    """
    df_sim = df.copy()
    baseline_revenue = df_sim['revenue'].sum()
    
    # Find rows eligible for a new promotion (currently have no promo)
    if target_categories:
        eligible_mask = (df_sim['category'].isin(target_categories)) & (df_sim['promo_flag'] == 0)
    else:
        eligible_mask = (df_sim['promo_flag'] == 0)
        
    eligible_indices = df_sim[eligible_mask].index

    num_to_convert = int(len(eligible_indices) * conversion_rate)
    
    if num_to_convert > 0:
        converted_indices = df_sim.loc[eligible_indices].sort_values(by='revenue', ascending=False).head(num_to_convert).index
        
        # Update promo markers
        df_sim.loc[converted_indices, 'promo_flag'] = 1
        df_sim.loc[converted_indices, 'promo_type'] = promo_type
        
        # Apply discount to price
        df_sim.loc[converted_indices, 'new_price'] = np.round(df_sim.loc[converted_indices, 'price'] * (1 - discount_pct), 2)
        
        # Simulate promotional demand spike using lift factor
        new_units = np.ceil(df_sim.loc[converted_indices, 'units_sold'] * lift_factor).astype(int)
        df_sim.loc[converted_indices, 'new_units_sold'] = new_units
        
        # Calculate revenue for the converted promo rows
        df_sim.loc[converted_indices, 'projected_revenue'] = np.round(df_sim.loc[converted_indices, 'new_units_sold'] * df_sim.loc[converted_indices, 'new_price'], 2)
        
    # Fill unconverted and non-targeted rows with their original baseline revenue
    df_sim['projected_revenue'] = df_sim['projected_revenue'].fillna(df_sim['revenue'])
    
    # Calculate total projected revenue across the entire portfolio
    projected_revenue = df_sim['projected_revenue'].sum()
    
    impact = {
        'baseline_revenue': round(baseline_revenue, 2),
        'projected_revenue': round(projected_revenue, 2),
        'revenue_change_pct': round(((projected_revenue - baseline_revenue) / baseline_revenue) * 100, 2),
        'discount_pct_used': discount_pct,
        'conversion_rate_used': conversion_rate,
        'lift_factor_used': lift_factor
    }
    
    return impact

In [11]:
simulate_promo_campaign(df, target_categories=['Beverages', 'Snacks'])

{'baseline_revenue': np.float64(337969.46),
 'projected_revenue': np.float64(348002.25),
 'revenue_change_pct': np.float64(2.97),
 'discount_pct_used': 0.2,
 'conversion_rate_used': 0.3,
 'lift_factor_used': 1.5}